### Members' Full name:

- Nguyen Mai Dinh, Le - 300312139
- Ruiz, Ricardo - 300387021
- Jimmy, Suwarly - 300361475
- Hugh, Tran - 300394597

# Part A. Planning

# Part B. Basic Model

### 1. Import Library + Data Loading (Demi)

In [1]:
# Import all libraries for Parts B, C and D in one cell
import os
import re
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display

# Part B
try:
    from elasticsearch import Elasticsearch
except Exception:
    Elasticsearch = None

# Part C
try:
    import spacy
    from sklearn.metrics.pairwise import cosine_similarity
except Exception:
    spacy = None
    cosine_similarity = None

In [2]:
df1 = pd.read_csv("faq.csv")
df2 = pd.read_csv("faq1.csv")
df3 = pd.read_csv("faq2.csv")
df4 = pd.read_csv("faq3.csv")
df5 = pd.read_csv("faq4.csv")
df6 = pd.read_csv("faq5.csv")
df7 = pd.read_csv("faq6.csv")
df8 = pd.read_csv("faq7.csv")

df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8], ignore_index=True)
df

,question,answer
0,How many programs can I apply to?,You can indicate one program choice per applic...
1,What is the fee to apply?,$37.40 for Canadian/Permanent Residents. $100 ...
2,What is preferential admission?,Preferential admission is a selective admissio...
3,How long is the wait-list for Douglas College ...,There is no wait-list for open enrolment or li...
4,I got a letter saying I'm in the program. Why ...,The first letter you received was an acknowled...
...,...,...
72,"If a student has to stay home (flu symptoms, f...",It is expected that we may have higher than no...
73,"If I am sick, are there alternatives to attend...",Decision regarding attendance is the discretio...
74,Will there be more options for online learning...,While Douglas College is returning to primaril...
75,Will I be able to access online classes / reco...,Not necessarily. Students should not have the ...


### 2. Search Engine Building (Ricardo)

In [3]:
# Connection settings
INDEX_NAME = "douglas_faq_basic"

ES_HOST = "localhost"
ES_PORT = 9200
ES_SCHEME = "https"
ES_USERNAME = "elastic"
ES_PASSWORD = "YYIT5POvXPBMFmS*1mB4"

es = None
es_ready = False

if Elasticsearch is None:
    print("Elasticsearch library is not installed in this environment.")
else:
    try:
        es = Elasticsearch(
            [{"host": ES_HOST, "port": ES_PORT, "scheme": ES_SCHEME}],
            basic_auth=(ES_USERNAME, ES_PASSWORD),
            verify_certs=False
        )
        if es.ping():
            es_ready = True
            print("Connected to Elasticsearch successfully.")
        else:
            print("Could not connect to Elasticsearch. Please make sure the local server is running.")
    except Exception as e:
        print("Elasticsearch client/server is not ready yet.")
        print("Details:", e)

Connected to Elasticsearch successfully.


In [4]:
# Create a new index
if es_ready:
    if es.indices.exists(index=INDEX_NAME):
        es.indices.delete(index=INDEX_NAME)

    mapping = {
        "mappings": {
            "properties": {
                "question": {"type": "text"},
                "answer": {"type": "text"}
            }
        }
    }

    es.indices.create(index=INDEX_NAME, body=mapping)
    print(f"Index '{INDEX_NAME}' created.")
else:
    print("Skipping index creation because Elasticsearch is not ready.")

Index 'douglas_faq_basic' created.


In [5]:
# Index all FAQ documents
if es_ready:
    for i, row in df.iterrows():
        doc = {
            "question": row["question"],
            "answer": row["answer"]
        }
        es.index(index=INDEX_NAME, id=i + 1, body=doc)

    es.indices.refresh(index=INDEX_NAME)
    res = es.search(index=INDEX_NAME, body={"query": {"match_all": {}}})
    print("Indexing complete.")
    print("Total documents in the index:", res["hits"]["total"]["value"])
else:
    print("Skipping document indexing because Elasticsearch is not ready.")

Indexing complete.
Total documents in the index: 77


In [6]:
# Search function for Part B
def search_faq(user_query, top_k=5):
    if not es_ready:
        return pd.DataFrame(columns=["rank", "score", "question", "answer"])

    body = {
        "size": top_k,
        "query": {
            "bool": {
                "should": [
                    {
                        "multi_match": {
                            "query": user_query,
                            "fields": ["question^2", "answer"]
                        }
                    },
                    {
                        "match_phrase": {
                            "question": {
                                "query": user_query,
                                "boost": 2
                            }
                        }
                    }
                ]
            }
        }
    }

    response = es.search(index=INDEX_NAME, body=body)

    rows = []
    for rank, hit in enumerate(response["hits"]["hits"], start=1):
        rows.append({
            "rank": rank,
            "score": hit["_score"],
            "question": hit["_source"]["question"],
            "answer": hit["_source"]["answer"]
        })

    return pd.DataFrame(rows)

In [ ]:
# Example search
sample_query = "How can I apply to  Douglas College?"
display(search_faq(sample_query, top_k=5))

### 3. Testing + Evaluation

#### 3.1. Precision@K Function + Testing + Evaluation function (Demi)

In [8]:
def run_queries(es, index_name, queries, top_k=5):
    results = {}
    
    for q in queries:
        response = es.search(
            index=index_name,
            body={
                "query": {
                    "multi_match": {
                        "query": q,
                        "fields": ["question^2", "answer"]
                    }
                },
                "size": top_k
            }
        )
        
        docs = []
        for hit in response["hits"]["hits"]:
            docs.append(hit["_source"])
        
        results[q] = docs
    
    return results

def precision_at_k(relevance_list, k=1):
    return sum(relevance_list[:k]) / k

def evaluate_precision_at_1(relevance_judgments):
    scores = []
    
    for rels in relevance_judgments.values():
        p1 = precision_at_k(rels, k=1)
        scores.append(p1)
    
    avg_p1 = sum(scores) / len(scores)
    
    return scores, avg_p1

def evaluate_precision_at_k(relevance_judgments, k=5):
    scores = []
    
    for rels in relevance_judgments.values():
        pk = precision_at_k(rels, k=k)
        scores.append(pk)
    
    avg_pk = sum(scores) / len(scores)
    
    return scores, avg_pk

#### 3.2. 20 questions + Evaluation (All)

**Demi**

5 questions

Testing using the pre-built testing function

Evaluation

**Ricardo**

5 questions:
 - "If I arrive late in Canada, can I study online first?"
 - "My health insurance will end soon. Who should I talk to?"
 - "I need help paying for school. Where can I ask about financial support?"
 - "If I get sick and miss class, how can I ask my instructor for help?"
 - "I want to transfer later to a university. Who can help me plan my courses?"

Testing using the pre-built testing function

Evaluation

In [9]:
# Testing questions

ricardo_questions = [
    "If I arrive late in Canada, can I study online first?",
    "My health insurance will end soon. Who should I talk to?",
    "I need help paying for school. Where can I ask about financial support?",
    "If I get sick and miss class, how can I ask my instructor for help?",
    "I want to transfer later to a university. Who can help me plan my courses?"
]

ricardo_results = []

for query in ricardo_questions:
    response = es.search(
        index=INDEX_NAME,
        query={
            "match": {
                "question": query
            }
        },
        size=1
    )

    hits = response["hits"]["hits"]

    if len(hits) > 0:
        top_hit = hits[0]["_source"]
        top_question = top_hit.get("question", "N/A")
        top_answer = top_hit.get("answer", "N/A")
    else:
        top_question = "No result found"
        top_answer = "No answer found"

    ricardo_results.append({
        "query": query,
        "top_question": top_question,
        "top_answer": top_answer
    })

# Ask for manual relevance with evidence
ricardo_relevance = {}

for item in ricardo_results:
    print("\n" + "="*80)
    print("Query:")
    print(item["query"])
    print("\nTop returned question:")
    print(item["top_question"])
    print("\nTop returned answer:")
    print(item["top_answer"][:500] + "..." if len(item["top_answer"]) > 500 else item["top_answer"])

    while True:
        user_input = input("\nIs this top result relevant? Enter 1 = Yes, 0 = No: ").strip()
        if user_input in ["1", "0"]:
            ricardo_relevance[item["query"]] = [int(user_input)]
            break
        else:
            print("Invalid input. Please enter only 1 or 0.")

# Evaluate
ricardo_p1_scores, ricardo_avg_p1 = evaluate_precision_at_1(ricardo_relevance)

print("\nRicardo relevance judgments:")
print(ricardo_relevance)
print("\nRicardo Precision@1 for each query:", ricardo_p1_scores)
print("Ricardo Average Precision@1:", ricardo_avg_p1)


Query:
If I arrive late in Canada, can I study online first?

Top returned question:
I am in Canada, can I take online courses in the Winter 2022 semester?

Top returned answer:
Starting from Winter 2022 semester we strongly recommend that students who are inside Canada to complete in-person classes to meet post-graduation work permit eligibility requirements. You can review Immigration, Refugees, and Citizenship Canada (IRCC) updates on distance learning here.



Is this top result relevant? Enter 1 = Yes, 0 = No:  0



Query:
My health insurance will end soon. Who should I talk to?

Top returned question:
My medical insurance is expiring. What should I do?

Top returned answer:
If your medical insurance is expiring, now is the time to ensure you are covered. For information on medical insurance for international students, please visit this page for options.



Is this top result relevant? Enter 1 = Yes, 0 = No:  1



Query:
I need help paying for school. Where can I ask about financial support?

Top returned question:
Can I ask someone if they've been immunized?

Top returned answer:
No. Health care information, including COVID-19 immunization, is considered private.


...


Query:
If I get sick and miss class, how can I ask my instructor for help?

Top returned question:
If I am sick, are there alternatives to attend class/take exams online?

Top returned answer:
Decision regarding attendance is the discretion of individual instructors, and faculty are encouraged to be reasonable when dealing with absences. Students continue to have the responsibility to communicate when illness impacts their ability to meet the requirements of a class.   It is expected that we may have higher than normal absenteeism this semester, particularly in the first few weeks of classes. Instructors will do their best to be understanding of students’ situations. Any resources or ...



Is this top result relevant? Enter 1 = Yes, 0 = No:  1



Query:
I want to transfer later to a university. Who can help me plan my courses?

Top returned question:
I'm interested in transferring to a university, how should I plan my courses?

Top returned answer:
Please review our University Transfer page for more information on how to plan for your transfer.  We encourage students to meet with a Student Success Advisor early to plan a successful transfer. 



Is this top result relevant? Enter 1 = Yes, 0 = No:  1



Ricardo relevance judgments:
{'If I arrive late in Canada, can I study online first?': [0], 'My health insurance will end soon. Who should I talk to?': [1], 'I need help paying for school. Where can I ask about financial support?': [0], 'If I get sick and miss class, how can I ask my instructor for help?': [1], 'I want to transfer later to a university. Who can help me plan my courses?': [1]}

Ricardo Precision@1 for each query: [0.0, 1.0, 0.0, 1.0, 1.0]
Ricardo Average Precision@1: 0.6


**Jimmy**

5 questions

Testing using the pre-built testing function

Evaluation

**Hugh**

5 questions:
- How can I submit my application?
- What do I need to pay for tuition?
- Where can I get advising help?
- How do I talk to a student advisor?
- What happens if I fail a course?

Testing using the pre-built testing function

Evaluation

In [10]:
# Testing questions - Hugh

hugh_questions = [
    "How can I submit my application?",
    "What do I need to pay for tuition?",
    "Where can I get advising help?",
    "How do I talk to a student advisor?",
    "What happens if I fail a course?"
]

hugh_results = []

for query in hugh_questions:
    response = es.search(
        index=INDEX_NAME,
        query={
            "match": {
                "question": query
            }
        },
        size=1
    )

    hits = response["hits"]["hits"]

    if len(hits) > 0:
        top_hit = hits[0]["_source"]
        top_question = top_hit.get("question", "N/A")
        top_answer = top_hit.get("answer", "N/A")
    else:
        top_question = "No result found"
        top_answer = "No answer found"

    hugh_results.append({
        "query": query,
        "top_question": top_question,
        "top_answer": top_answer
    })

# Ask for manual relevance with evidence
hugh_relevance = {}

for item in hugh_results:
    print("\n" + "="*80)
    print("Query:")
    print(item["query"])
    print("\nTop returned question:")
    print(item["top_question"])
    print("\nTop returned answer:")
    print(item["top_answer"][:500] + "..." if len(item["top_answer"]) > 500 else item["top_answer"])

    while True:
        user_input = input("\nIs this top result relevant? Enter 1 = Yes, 0 = No: ").strip()
        if user_input in ["1", "0"]:
            hugh_relevance[item["query"]] = [int(user_input)]
            break
        else:
            print("Invalid input. Please enter only 1 or 0.")

# Evaluate
hugh_p1_scores, hugh_avg_p1 = evaluate_precision_at_1(hugh_relevance)

print("\nHugh relevance judgments:")
print(hugh_relevance)
print("\nHugh Precision@1 for each query:", hugh_p1_scores)
print("Hugh Average Precision@1:", hugh_avg_p1)


Query:
How can I submit my application?

Top returned question:
How will COVID-19 impact processing of my immigration applications (study/work permits, permanent residence application, biometrics, etc.)?

Top returned answer:
To find the latest information on Immigration, Refugees and Citizenship Canada’s policy and service adjustments due to COVID-19, please review the International Student IRCC Coronavirus Disease: International Student page.



Is this top result relevant? Enter 1 = Yes, 0 = No:  0



Query:
What do I need to pay for tuition?

Top returned question:
Do I need to prepare anything for my appointment?

Top returned answer:
No, you do not need to prepare anything. However, we advise you to read the Information and Consent to Counsel Form prior to your session. Your counsellor will go over this when you arrive, so it will speed up the process if you are already familiar with it.



Is this top result relevant? Enter 1 = Yes, 0 = No:  0



Query:
Where can I get advising help?

Top returned question:
Where can I get more information on student loans, grants, awards, scholarships, and bursaries?

Top returned answer:
Please contact our Financial Aid department to get detailed information about these topics. 


...


Query:
How do I talk to a student advisor?

Top returned question:
I am almost done my program. How do I apply to graduate?

Top returned answer:
You should apply to graduate after you have enrolled in your final term of courses. You can apply by completing the online graduation application found on myAccount under "Student Records". Learn more from our graduation page.



Is this top result relevant? Enter 1 = Yes, 0 = No:  0



Query:
What happens if I fail a course?

Top returned question:
What happens if someone on campus gets COVID?

Top returned answer:
The Return to Campus Guidelines highlights that while eliminating the COVID-19 virus will not occur in the near future, we can certainly adapt to living with COVID-19 as we do with other manageable seasonal ailments such as influenza. If someone in the community does contract COVID, the College will follow the public health guidelines provided by the PHO for reporting and notification. Preventing the spread of COVID-19 relies on everyone doing their part, including: immunization, daily self-admi...



Is this top result relevant? Enter 1 = Yes, 0 = No:  0



Hugh relevance judgments:
{'How can I submit my application?': [0], 'What do I need to pay for tuition?': [0], 'Where can I get advising help?': [0], 'How do I talk to a student advisor?': [0], 'What happens if I fail a course?': [0]}

Hugh Precision@1 for each query: [0.0, 0.0, 0.0, 0.0, 0.0]
Hugh Average Precision@1: 0.0


In [ ]:
group_relevance_b = {}
group_relevance_b.update(demi_relevance)
group_relevance_b.update(ricardo_relevance)
group_relevance_b.update(jimmy_relevance)
group_relevance_b.update(hugh_relevance)

if len(group_relevance_b) > 0:
    group_scores_b, group_avg_b = evaluate_precision_at_1(group_relevance_b)
    print("Total questions counted:", len(group_relevance_b))
    print("Group Precision@1 scores:", group_scores_b)
    print("Group Average Precision@1:", group_avg_b)
else:
    print("Please add all members' relevance judgments first.")

# Part C. ElasticSearch with Embedding

### 1. Discussion (Demi)

### 2. Building Embedding (Hugh)

In [14]:
# pip install elasticsearch
# %pip install -U spacy
# !python -m spacy download en_core_web_lg

import re
import spacy
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from elasticsearch import Elasticsearch
# warnings.filterwarnings("ignore")

es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic", "YYIT5POvXPBMFmS*1mB4"),
    verify_certs=False
)
print(es.info())

{'name': 'RIDOR', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'eEp0XNSGSJylgHnpklR4Lw', 'version': {'number': '9.3.2', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '43a703737aab6baefa748bc7b69e4054926f2b2c', 'build_date': '2026-03-16T13:12:56.143057855Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [ ]:
nlp = spacy.load("en_core_web_lg")

In [17]:
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["question_clean"] = df["question"].apply(clean_text)
df["answer_clean"] = df["answer"].apply(clean_text)
# df.info()

In [18]:
def get_embedding(text, nlp_model):
    return nlp_model(text).vector

df["question_vector"] = df["question_clean"].apply(lambda x: get_embedding(x, nlp))
# df["question_vector"]

In [19]:
faq_matrix = np.vstack(df["question_vector"].values)
faq_matrix.shape

(77, 96)

In [20]:
def semantic_search(query, df, faq_matrix, nlp_model, top_k=5):
    query_clean = clean_text(query)
    query_vector = nlp_model(query_clean).vector.reshape(1, -1)

    similarities = cosine_similarity(query_vector, faq_matrix)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[top_indices].copy()
    results["similarity_score"] = similarities[top_indices]

    return results[["question", "answer", "similarity_score"]]

In [21]:
test_query = "How can I apply to Douglas?"
semantic_search(test_query, df, faq_matrix, nlp, top_k=5)

,question,answer,similarity_score
8,Am I allowed to travel to Canada?,Travel exemptions and restrictions are constan...,0.706540
49,I am almost done my program. How do I apply to...,You should apply to graduate after you have en...,0.689106
51,I'm interested in transferring to a university...,Please review our University Transfer page for...,0.684458
11,How will my PGWP eligibility be impacted by CO...,Students may count all their time studying onl...,0.684168
35,Can I transfer tuition fees?,Tuition fees are not transferable to another s...,0.668327


In [22]:
def run_semantic_queries(df, faq_matrix, queries, nlp_model, top_k=5):
    results = {}

    for q in queries:
        result_df = semantic_search(q, df, faq_matrix, nlp_model, top_k=top_k)
        results[q] = result_df

    return results

### 2. Testing + Evaluation (All)

In [23]:
hugh_test_questions = [
    "How can I submit my application?",
    "What do I need to pay for tuition?",
    "Where can I get advising help?",
    "How do I talk to a student advisor?",
    "What happens if I fail a course?"
]

In [24]:
semantic_results = run_semantic_queries(df, faq_matrix, hugh_test_questions, nlp, top_k=5)

for q, res in semantic_results.items():
    print("=" * 100)
    print("QUESTION:", q)
    print(res)
    print()

QUESTION: How can I submit my application?
                                             question  \
61              How quickly can I get an appointment?   
35                       Can I transfer tuition fees?   
56            What can I expect in the first meeting?   
49  I am almost done my program. How do I apply to...   
48  How can I find out more information about a pr...   

                                               answer  similarity_score  
61  Counselling appointments are typically booked ...          0.733022  
35  Tuition fees are not transferable to another s...          0.724818  
56  You’ll begin your meeting with a review of our...          0.707897  
49  You should apply to graduate after you have en...          0.705658  
48  You can attend an information session or you c...          0.663322  

QUESTION: What do I need to pay for tuition?
                                             question  \
67  What information do I need to provide to book ...   
62  Do I n

In [25]:
# Ricardo - Part C testing + manual evaluation

ricardo_questions_c = [
    "If I arrive late in Canada, can I study online first?",
    "My health insurance will end soon. Who should I talk to?",
    "I need help paying for school. Where can I ask about financial support?",
    "If I get sick and miss class, how can I ask my instructor for help?",
    "I want to transfer later to a university. Who can help me plan my courses?"
]

ricardo_results_c = []

for query in ricardo_questions_c:
    results_df = semantic_search(query, df, faq_matrix, nlp, top_k=5)

    if len(results_df) > 0:
        top_row = results_df.iloc[0]
        top_question = top_row["question"]
        top_answer = top_row["answer"]
        top_score = top_row["similarity_score"]
    else:
        top_question = "No result found"
        top_answer = "No answer found"
        top_score = 0

    ricardo_results_c.append({
        "query": query,
        "top_question": top_question,
        "top_answer": top_answer,
        "similarity_score": top_score
    })

# Ask for manual relevance with evidence
ricardo_relevance_c = {}

for item in ricardo_results_c:
    print("\n" + "=" * 90)
    print("Query:")
    print(item["query"])
    print("\nTop returned question:")
    print(item["top_question"])
    print("\nTop returned answer:")
    print(item["top_answer"][:500] + "..." if len(item["top_answer"]) > 500 else item["top_answer"])
    print("\nSimilarity score:")
    print(item["similarity_score"])

    while True:
        user_input = input("\nIs this top result relevant? Enter 1 = Yes, 0 = No: ").strip()
        if user_input in ["1", "0"]:
            ricardo_relevance_c[item["query"]] = [int(user_input)]
            break
        else:
            print("Invalid input. Please enter only 1 or 0.")

# Evaluate Precision@1
ricardo_p1_scores_c, ricardo_avg_p1_c = evaluate_precision_at_1(ricardo_relevance_c)

print("\nRicardo Part C relevance judgments:")
print(ricardo_relevance_c)
print("\nRicardo Part C Precision@1 scores:", ricardo_p1_scores_c)
print("Ricardo Part C Average Precision@1:", ricardo_avg_p1_c)

ricardo_results_c_df = pd.DataFrame(ricardo_results_c)
display(ricardo_results_c_df)


Query:
If I arrive late in Canada, can I study online first?

Top returned question:
How will my PGWP eligibility be impacted by COVID-19 if I study outside of Canada?

Top returned answer:
Students may count all their time studying online outside Canada from Winter 2020 until August 31, 2022, according to the recently updated rules posted on the IRCC website. You’re eligible for this temporary policy if you meet all of the following criteria: You’re enrolled in a PGWP-eligible program. You were outside Canada and unable to travel to Canada because of the COVID-19 pandemic but were still able to take online courses. You began a program in any semester from spring 2020 to summer 202...

Similarity score:
0.6893103



Is this top result relevant? Enter 1 = Yes, 0 = No:  0



Query:
My health insurance will end soon. Who should I talk to?

Top returned question:
My medical insurance is expiring. What should I do?

Top returned answer:
If your medical insurance is expiring, now is the time to ensure you are covered. For information on medical insurance for international students, please visit this page for options.

Similarity score:
0.75288045



Is this top result relevant? Enter 1 = Yes, 0 = No:  1



Query:
I need help paying for school. Where can I ask about financial support?

Top returned question:
What can I talk about in my appointment? Can I talk about several topics in one session?

Top returned answer:
You are welcome to bring up any topic of concern, and in some cases seemingly unrelated topics may be intertwined. Your counsellor will let you know if something is beyond their scope of practice and support you in finding resources elsewhere.

Similarity score:
0.7557954


...


Is this top result relevant? Enter 1 = Yes, 0 = No:  0



Query:
I want to transfer later to a university. Who can help me plan my courses?

Top returned question:
Do I need to prepare anything for my appointment?

Top returned answer:
No, you do not need to prepare anything. However, we advise you to read the Information and Consent to Counsel Form prior to your session. Your counsellor will go over this when you arrive, so it will speed up the process if you are already familiar with it.

Similarity score:
0.75836736



Is this top result relevant? Enter 1 = Yes, 0 = No:  0



Ricardo Part C relevance judgments:
{'If I arrive late in Canada, can I study online first?': [0], 'My health insurance will end soon. Who should I talk to?': [1], 'I need help paying for school. Where can I ask about financial support?': [0], 'If I get sick and miss class, how can I ask my instructor for help?': [0], 'I want to transfer later to a university. Who can help me plan my courses?': [0]}

Ricardo Part C Precision@1 scores: [0.0, 1.0, 0.0, 0.0, 0.0]
Ricardo Part C Average Precision@1: 0.2


,query,top_question,top_answer,similarity_score
0,"If I arrive late in Canada, can I study online...",How will my PGWP eligibility be impacted by CO...,Students may count all their time studying onl...,0.689310
1,My health insurance will end soon. Who should ...,My medical insurance is expiring. What should ...,"If your medical insurance is expiring, now is ...",0.752880
2,I need help paying for school. Where can I ask...,What can I talk about in my appointment? Can I...,You are welcome to bring up any topic of conce...,0.755795
3,"If I get sick and miss class, how can I ask my...",Do I need a study permit to take classes if th...,Yes. You are required to have a valid study pe...,0.653560
4,I want to transfer later to a university. Who ...,Do I need to prepare anything for my appointment?,"No, you do not need to prepare anything. Howev...",0.758367


# Part D. FAQBot using Gemini

### 1. RAG Building (Jimmy)

### 2. Testing + Evaluation (All)

# Part E. Final Conclusion + Compare Results (Demi)